In [1]:
"""
Classical optimizers in VQE for a two-site lattice pionless EFT deuteron.

P. Çifci and S. Akkoyun, Cumhuriyet Science Journal (2026).
Hamiltonian and constants as in Çifci, Akkoyun and La Ronde, Eur. Phys. J. A 62, 199 (2026).

    python csj_vqe.py                    # all runs, numpy engine
    python csj_vqe.py --engine qiskit    # energies from Qiskit (slow)
    python csj_vqe.py --workers 4        # parallel runs (terminal only)
    python csj_vqe.py --appendix         # Appendix B tables from saved runs

Output goes to csj_out_<engine>/. Runs already in runs.jsonl are skipped.
"""
import argparse
import csv
import json
import os
import platform
import sys
import time
import warnings
from collections import Counter

import numpy as np
import scipy
import qiskit
import qiskit_nature
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper

warnings.filterwarnings('ignore')

HBAR_C = 197.3                              # MeV fm
M_N = 940.0                                 # MeV
A_FM = 4.5                                  # fm
T_HOP = HBAR_C**2 / (2 * M_N * A_FM**2)     # 1.0225 MeV
C_CONTACT = -0.344568                       # MeV, from the brentq fit in the ED script
E_ED = -2.224566                            # MeV

OPTIMIZERS = ['COBYLA', 'Nelder-Mead', 'Powell', 'SLSQP', 'L-BFGS-B']
SEEDS = [42, 123, 456, 789, 1024, 7, 11, 19, 23, 31,
         37, 53, 61, 71, 83, 97, 101, 113, 127, 131]
INITS = {'narrow': (0.1, 0.5),
         'near0': (-0.1, 0.1),
         'pm_pi': (-np.pi, np.pi),
         'zero_2pi': (0.0, 2 * np.pi)}
CONV_TOL = 0.01         # MeV
W_MIN = 0.99            # minimum (1p,1n) weight of a converged run
FREE_CAP = 20000        # evaluation limit in the free protocol
SPSA_BUDGET = 2000
SPSA_STEP = 0.1         # rad, size of the first SPSA update
SPSA_C = 0.1            # rad
ENGINE_TOL = 1e-9       # MeV
SECTOR_VAR_TOL = 1e-3   # var(Np) and var(Nn) below this: pure sector (also in csj_figs.py)

ROT_QUBITS = (1, 3, 5, 7)
PAIRS = ((1, 5), (3, 7), (1, 3), (5, 7))


# Hamiltonian and ansatz

def mode(site, sigma):
    # site-major ordering, sigma = 0, 1, 2, 3 for p up, p down, n up, n down
    return 4 * site + sigma


def build_hamiltonian():
    terms = {}
    for s in range(4):
        i, j = mode(0, s), mode(1, s)
        terms[f"+_{i} -_{j}"] = -T_HOP
        terms[f"+_{j} -_{i}"] = -T_HOP
    for site in (0, 1):
        for sp in (0, 1):
            for sn in (2, 3):
                p, n = mode(site, sp), mode(site, sn)
                terms[f"+_{p} -_{p} +_{n} -_{n}"] = C_CONTACT   # C for each pn pair
    return JordanWignerMapper().map(FermionicOp(terms, num_spin_orbitals=8))


def build_ansatz(depth):
    qc = QuantumCircuit(8)
    qc.x(1)
    qc.x(3)
    th = ParameterVector('θ', 8 * depth)
    for layer in range(depth):
        o = 8 * layer
        for k, q in enumerate(ROT_QUBITS):
            qc.ry(th[o + k], q)
        for k, (c, t) in enumerate(PAIRS):
            qc.cx(c, t)
            qc.ry(th[o + 4 + k], t)
            qc.cx(c, t)
    return qc


def bind(ansatz, x):
    return ansatz.assign_parameters(dict(zip(ansatz.parameters, x)))


def particle_numbers():
    idx = np.arange(256)
    Np = sum((idx >> k) & 1 for k in (0, 1, 4, 5))
    Nn = sum((idx >> k) & 1 for k in (2, 3, 6, 7))
    return Np, Nn


class NumpyEngine:
    # The gates are real, so the state vector stays real. Qubit q is bit q of the index.

    def __init__(self, H, depth):
        self.H = H
        self.depth = depth
        idx = np.arange(256)
        self.perm = {(c, t): idx ^ (((idx >> c) & 1) << t) for c, t in PAIRS}

    @staticmethod
    def ry(psi, theta, q):
        p = psi.reshape((2,) * 8)
        ax = 7 - q
        a0, a1 = np.take(p, 0, axis=ax), np.take(p, 1, axis=ax)
        c, s = np.cos(theta / 2), np.sin(theta / 2)
        return np.stack([c * a0 - s * a1, s * a0 + c * a1], axis=ax).reshape(256)

    def state(self, x):
        psi = np.zeros(256)
        psi[0b1010] = 1.0
        for layer in range(self.depth):
            o = 8 * layer
            for k, q in enumerate(ROT_QUBITS):
                psi = self.ry(psi, x[o + k], q)
            for k, (c, t) in enumerate(PAIRS):
                perm = self.perm[(c, t)]
                psi = self.ry(psi[perm], x[o + 4 + k], t)[perm]
        return psi

    def energy(self, x):
        psi = self.state(x)
        return float(psi @ self.H @ psi)


_cache = {}


def setup(depth):
    if depth not in _cache:
        H = build_hamiltonian()
        H_mat = H.to_matrix()
        assert np.abs(H_mat.imag).max() < 1e-10
        H_mat = H_mat.real
        Np, Nn = particle_numbers()
        _cache[depth] = dict(H=H, H2=H.compose(H).simplify(), H_mat=H_mat,
                             ansatz=build_ansatz(depth), est=StatevectorEstimator(),
                             Np=Np, Nn=Nn, phys=(Np == 1) & (Nn == 1),
                             fast=NumpyEngine(H_mat, depth))
    return _cache[depth]


def qiskit_energy(x, ctx):
    job = ctx['est'].run([(bind(ctx['ansatz'], x), ctx['H'])])
    return float(job.result()[0].data.evs.real)


# Checks

def engine_check(n=20):
    ctx = setup(2)
    rng = np.random.RandomState(12345)
    worst = 0.0
    for _ in range(n):
        x = rng.uniform(-np.pi, np.pi, 16)
        worst = max(worst, abs(ctx['fast'].energy(x) - qiskit_energy(x, ctx)))
    print(f"numpy vs Qiskit energy, largest difference over {n} points: {worst:.2e} MeV")
    return worst < ENGINE_TOL


def check_setup(out):
    os.makedirs(out, exist_ok=True)
    env = [f"platform {platform.platform()}",
           f"cpu {platform.processor()}",
           f"logical cores {os.cpu_count()}",
           f"python {platform.python_version()}",
           f"numpy {np.__version__}",
           f"scipy {scipy.__version__}",
           f"qiskit {qiskit.__version__}",
           f"qiskit_nature {qiskit_nature.__version__}"]
    try:
        import psutil
        env.append(f"RAM {psutil.virtual_memory().total / 1e9:.1f} GB")
    except ImportError:
        pass
    with open(os.path.join(out, 'environment.txt'), 'w', encoding='utf-8') as f:
        f.write("\n".join(env) + "\n")
    print("\n".join(env))

    ctx = setup(2)
    H, Np, Nn = ctx['H_mat'], ctx['Np'], ctx['Nn']
    e0 = np.linalg.eigvalsh(H[np.ix_(ctx['phys'], ctx['phys'])])[0]
    print(f"\nt = {T_HOP:.6f} MeV, C = {C_CONTACT} MeV")
    print(f"lowest (1p,1n) eigenvalue = {e0:.8f} MeV (E_ED = {E_ED} MeV)")
    if abs(e0 - E_ED) > 1e-5:
        sys.exit("Hamiltonian does not reproduce E_ED")

    # the ansatz does not act on the spin-up qubits 0, 2, 4, 6
    down = (np.arange(256) & 0b01010101) == 0
    print("lowest energy of each sector in the spin-down subspace:")
    for p in range(3):
        for n in range(3):
            m = down & (Np == p) & (Nn == n)
            print(f"  ({p}p,{n}n)  {np.linalg.eigvalsh(H[np.ix_(m, m)])[0]:9.5f} MeV")

    op = ctx['H'].simplify()
    with open(os.path.join(out, 'hamiltonian_pauli.csv'), 'w', encoding='utf-8') as f:
        f.write('pauli_q7_to_q0,coefficient_MeV\n')
        for label, c in zip(op.paulis.to_labels(), op.coeffs):
            f.write(f'{label},{c.real:.12f}\n')
    print(f"{len(op)} Pauli terms written to hamiltonian_pauli.csv")

    if not engine_check():
        sys.exit("numpy engine does not match Qiskit")
    x = np.full(16, 0.3)
    t0 = time.perf_counter()
    for _ in range(200):
        ctx['fast'].energy(x)
    t_np = (time.perf_counter() - t0) / 200
    t0 = time.perf_counter()
    for _ in range(20):
        qiskit_energy(x, ctx)
    t_q = (time.perf_counter() - t0) / 20
    print(f"time per energy: numpy {t_np * 1e3:.2f} ms, Qiskit {t_q * 1e3:.1f} ms\n")


# Optimization

class BudgetExceeded(Exception):
    pass


def optimizer_options(opt):
    cap = FREE_CAP
    return {'COBYLA':      {'maxiter': cap, 'rhobeg': 1.0, 'tol': 1e-6},
            'Nelder-Mead': {'maxiter': cap, 'maxfev': cap, 'xatol': 1e-6, 'fatol': 1e-6},
            'Powell':      {'maxiter': cap, 'maxfev': cap, 'xtol': 1e-6, 'ftol': 1e-6},
            'SLSQP':       {'maxiter': cap, 'ftol': 1e-9},
            'L-BFGS-B':    {'maxiter': cap, 'maxfun': cap, 'ftol': 1e-12, 'gtol': 1e-8}}[opt]


def run_spsa(f, x0, rng, step, c0, budget):
    # Spall (1992). a is chosen from 25 gradient estimates at x0 so that the first
    # update has size `step`. The loop ends when f raises BudgetExceeded.
    n = len(x0)
    alpha, gamma = 0.602, 0.101
    A = 0.1 * (budget // 2)
    mags = []
    for _ in range(25):
        d = rng.choice([-1.0, 1.0], n)
        mags.append(abs(f(x0 + c0 * d) - f(x0 - c0 * d)) / (2 * c0))
    a = step * (A + 1)**alpha / max(np.mean(mags), 1e-12)
    x = np.array(x0, dtype=float)
    k = 0
    while True:
        ak = a / (A + k + 1)**alpha
        ck = c0 / (k + 1)**gamma
        d = rng.choice([-1.0, 1.0], n)
        x = x - ak * (f(x + ck * d) - f(x - ck * d)) / (2 * ck) * d
        k += 1
        if k % 20 == 0:
            f(x)


def optimize(job, ctx):
    budget = job.get('budget', FREE_CAP)
    lo, hi = INITS[job['init']]
    x0 = np.random.RandomState(job['seed']).uniform(lo, hi, ctx['ansatz'].num_parameters)
    if job['engine'] == 'numpy':
        energy = ctx['fast'].energy
    else:
        energy = lambda x: qiskit_energy(x, ctx)

    hist = []
    best = {'e': np.inf, 'x': None}

    def f(x):
        if len(hist) >= budget:
            raise BudgetExceeded
        e = energy(x)
        hist.append(e)
        if e < best['e']:
            best['e'], best['x'] = e, np.array(x, dtype=float)
        return e

    term, message, nit, success, error = 'tolerance', '', None, None, None
    t0 = time.perf_counter()
    try:
        if job['opt'] == 'SPSA':
            run_spsa(f, x0, np.random.RandomState(job['seed'] + 10000),
                     SPSA_STEP, SPSA_C, budget)
        else:
            res = minimize(f, x0, method=job['opt'], options=optimizer_options(job['opt']))
            nit = int(res.nit) if getattr(res, 'nit', None) is not None else None
            success, message = bool(res.success), str(res.message)
    except BudgetExceeded:
        term = 'budget'
    except Exception as e:
        term, error = 'error', repr(e)
    elapsed = time.perf_counter() - t0
    if term == 'tolerance' and len(hist) >= budget:
        term = 'budget'
    return dict(hist=hist, best=best, term=term, message=message, nit=nit,
                success=success, error=error, elapsed=elapsed)


def run_job(job):
    ctx = setup(job['depth'])
    r = optimize(job, ctx)
    hist, best = r['hist'], r['best']
    rec = dict(job, term=r['term'], message=r['message'], error=r['error'], nit=r['nit'],
               success=r['success'], nfev=len(hist), elapsed=r['elapsed'])
    if not hist:
        return rec

    h = np.array(hist)
    bsf = np.minimum.accumulate(h)
    reached = np.nonzero(bsf < E_ED + CONV_TOL)[0]

    # quantities of the final state are recomputed with Qiskit
    Np, Nn = ctx['Np'], ctx['Nn']
    sv = Statevector.from_instruction(bind(ctx['ansatz'], best['x']))
    probs = sv.probabilities()
    E_q = float(sv.expectation_value(ctx['H']).real)
    mean_p, mean_n = float(probs @ Np), float(probs @ Nn)
    rec.update(
        E=float(best['e']),
        dE=float(best['e'] - E_ED),
        engine_diff=abs(float(best['e']) - E_q),
        var_H=float(sv.expectation_value(ctx['H2']).real - E_q**2),
        Np=mean_p, var_Np=max(0.0, float(probs @ Np**2) - mean_p**2),
        Nn=mean_n, var_Nn=max(0.0, float(probs @ Nn**2) - mean_n**2),
        w_phys=float(probs[ctx['phys']].sum()),
        evals_to_10keV=int(reached[0] + 1) if len(reached) else None,
        improve_last500=float(bsf[-501] - bsf[-1]) if len(h) > 500 else None,
        tail_range200=float(h[-200:].max() - h[-200:].min()),
        best_x=[float(v) for v in best['x']],
    )
    hist_dir = os.path.join(job['out'], 'hist')
    os.makedirs(hist_dir, exist_ok=True)
    fname = f"{job['init']}_{job['protocol']}_d{job['depth']}_{job['opt']}_{job['seed']}.npy"
    np.save(os.path.join(hist_dir, fname), h)
    rec['hist_file'] = fname
    return rec


def job_key(j):
    return "|".join(str(j[k]) for k in ('engine', 'init', 'protocol', 'depth', 'opt', 'seed'))


def make_jobs(nseeds, out, engine, inits=None, depth=2, opts=None, protocol='free', budget=None):
    jobs = []
    for init in inits or INITS:
        for opt in opts or OPTIMIZERS:
            for seed in SEEDS[:nseeds]:
                job = dict(init=init, protocol=protocol, depth=depth, opt=opt, seed=seed,
                           engine=engine, out=out)
                if budget:
                    job['budget'] = budget
                jobs.append(job)
    return jobs


def run_line(r):
    if 'E' not in r:
        return f"{r['init']:>8} {r['opt']:<11} seed={r['seed']:4d} ERROR {r['error']}"
    return (f"{r['init']:>8} {r['opt']:<11} d={r['depth']} seed={r['seed']:4d} "
            f"E={r['E']:9.5f} dE={r['dE']:+.2e} w={r['w_phys']:.4f} "
            f"nfev={r['nfev']:5d} {r['term']:<9} {r['elapsed']:6.1f}s")


def execute(jobs, out, workers):
    path = os.path.join(out, 'runs.jsonl')
    done = set()
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            done = {job_key(json.loads(line)) for line in f if line.strip()}
    jobs = [j for j in jobs if job_key(j) not in done]
    print(f"{len(jobs)} runs to do, {len(done)} already in {path}")
    t0 = time.time()
    with open(path, 'a', encoding='utf-8') as f:
        if workers > 1:
            from multiprocessing import Pool
            with Pool(workers) as pool:
                for n, r in enumerate(pool.imap_unordered(run_job, jobs), 1):
                    f.write(json.dumps(r) + "\n")
                    f.flush()
                    print(f"{n}/{len(jobs)} {run_line(r)}", flush=True)
        else:
            for n, job in enumerate(jobs, 1):
                r = run_job(job)
                f.write(json.dumps(r) + "\n")
                f.flush()
                print(f"{n}/{len(jobs)} {run_line(r)}", flush=True)
    print(f"done in {time.time() - t0:.0f} s\n")


# Analysis

def load_runs(out):
    with open(os.path.join(out, 'runs.jsonl'), encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return [r for r in rows if r['init'] in INITS]


def iqr(vals):
    if len(vals) == 0:
        return "-"
    q1, med, q3 = np.percentile(vals, [25, 50, 75])
    return f"{med:.0f} [{q1:.0f}-{q3:.0f}]"


def pct(vals, p):
    return float(np.percentile(vals, p)) if len(vals) else None


def classify(r):
    # same rule as in csj_figs.py
    if r['dE'] < CONV_TOL and r['w_phys'] > W_MIN:
        return 'conv'
    if r['var_Np'] < SECTOR_VAR_TOL and r['var_Nn'] < SECTOR_VAR_TOL:
        s = (round(r['Np']), round(r['Nn']))
        return {(2, 1): 's21', (0, 1): 's01'}.get(s, 'sother')
    return 'mixed'


def final_states(rows):
    leaks = [r for r in rows if r['w_phys'] <= W_MIN]
    print(f"runs outside the physical sector: {len(leaks)} / {len(rows)}")
    pure, mixed = {}, []
    for r in leaks:
        if r['var_Np'] < SECTOR_VAR_TOL and r['var_Nn'] < SECTOR_VAR_TOL:
            pure.setdefault((round(r['Np']), round(r['Nn'])), []).append(r)
        else:
            mixed.append(r)
    for (p, n), rs in sorted(pure.items()):
        es = [r['E'] for r in rs]
        print(f"  ({p}p,{n}n): {len(rs)} runs, E from {min(es):.5f} to {max(es):.5f} MeV, "
              f"max Var(H) = {max(r['var_H'] for r in rs):.1e} MeV^2")
    if mixed:
        print(f"  mixed states: {len(mixed)} runs")
        for r in mixed:
            print(f"    {r['init']:>8} {r['opt']:<11} seed={r['seed']:4d} E={r['E']:9.5f} "
                  f"<Np>={r['Np']:.3f} <Nn>={r['Nn']:.3f} w={r['w_phys']:.4f} "
                  f"Var(H)={r['var_H']:.2e}")


def summary(out):
    rows = load_runs(out)
    ok = [r for r in rows if 'E' in r]

    cols = ['engine', 'init', 'protocol', 'depth', 'opt', 'seed', 'term', 'nit', 'nfev',
            'elapsed', 'E', 'dE', 'engine_diff', 'var_H', 'Np', 'var_Np', 'Nn', 'var_Nn',
            'w_phys', 'evals_to_10keV', 'improve_last500', 'tail_range200', 'success', 'error']
    with open(os.path.join(out, 'runs_flat.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=cols, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)

    stats = []
    for depth, protocol in sorted({(r['depth'], r['protocol']) for r in ok}):
        grp = [r for r in ok if r['depth'] == depth and r['protocol'] == protocol]
        print("=" * 100)
        print(f"depth {depth}, protocol {protocol}. converged: dE < 10 keV and w > {W_MIN}. "
              f"values: median [Q1-Q3]")
        print("=" * 100)
        for init, (lo, hi) in INITS.items():
            sub = [r for r in grp if r['init'] == init]
            if not sub:
                continue
            print(f"\n{init} [{lo:.3f}, {hi:.3f}]")
            print(f"{'optimizer':<12}{'n':>3}{'conv':>5}{'leak':>5}{'cap':>5}   "
                  f"{'N_term':<22}{'mean +- std':<16}{'N_10keV (reached/n)'}")
            for opt in OPTIMIZERS + ['SPSA']:
                s = [r for r in sub if r['opt'] == opt]
                if not s:
                    continue
                n = len(s)
                conv = sum(r['dE'] < CONV_TOL and r['w_phys'] > W_MIN for r in s)
                leak = sum(r['w_phys'] <= W_MIN for r in s)
                cap = sum(r['term'] == 'budget' or r['nfev'] >= FREE_CAP for r in s)
                nf = np.array([r['nfev'] for r in s], dtype=float)
                t10 = [r['evals_to_10keV'] for r in s if r['evals_to_10keV'] is not None]
                sd = nf.std(ddof=1) if n > 1 else 0.0
                print(f"{opt:<12}{n:>3}{conv:>5}{leak:>5}{cap:>5}   {iqr(nf):<22}"
                      f"{nf.mean():>6.0f} +- {sd:<6.0f}  {iqr(t10)} ({len(t10)}/{n})")
                stats.append(dict(depth=depth, protocol=protocol, init=init, opt=opt, n=n,
                                  converged=conv, leaked=leak, budget_hits=cap,
                                  nfev_median=pct(nf, 50), nfev_q1=pct(nf, 25),
                                  nfev_q3=pct(nf, 75), nfev_mean=float(nf.mean()),
                                  nfev_std=float(sd), t10_median=pct(t10, 50),
                                  t10_q1=pct(t10, 25), t10_q3=pct(t10, 75),
                                  t10_reached=len(t10)))
        print()
        final_states(grp)
        print()

    if stats:
        with open(os.path.join(out, 'summary_stats.csv'), 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(stats[0]))
            w.writeheader()
            w.writerows(stats)

    diffs = [r['engine_diff'] for r in ok]
    if diffs:
        print(f"largest numpy/Qiskit energy difference at final points: {max(diffs):.2e} MeV")
    for r in rows:
        if r['term'] == 'error':
            print(f"ERROR {r['init']} {r['opt']} seed={r['seed']}: {r['error']}")


def verify_prefix(out, engine):
    # A run limited to 2000 evaluations should repeat the first 2000 evaluations of
    # the free run, so that fixed-budget results can be read from the free histories.
    ctx = setup(2)
    all_same = True
    for opt in OPTIMIZERS:
        for init, seed in (('narrow', 42), ('pm_pi', 123)):
            ref = np.load(os.path.join(out, 'hist', f"{init}_free_d2_{opt}_{seed}.npy"))
            job = dict(init=init, protocol='verify', depth=2, opt=opt, seed=seed,
                       engine=engine, out=out, budget=2000)
            h = np.array(optimize(job, ctx)['hist'])
            m = min(len(ref), 2000)
            same = len(h) == m and np.allclose(h, ref[:m], rtol=0, atol=1e-12)
            all_same &= bool(same)
            print(f"  {opt:<12} {init:<7} seed={seed}: {'same' if same else 'DIFFERENT'}")
    return all_same


def budget_report(out, engine):
    print("prefix check, budget 2000 against the free runs:")
    if not verify_prefix(out, engine):
        print("prefix check failed, the fixed-budget table below is not valid")
    rows = [r for r in load_runs(out) if 'E' in r and r['depth'] == 2 and
            (r['protocol'] == 'free' or (r['protocol'] == 'fixed' and r['opt'] == 'SPSA'))]
    budgets = (500, 1000, 2000)
    print("\nruns within 10 keV of E_ED after B evaluations")
    for init in INITS:
        sub = [r for r in rows if r['init'] == init]
        if not sub:
            continue
        print(f"\n{init}")
        print(f"{'optimizer':<12}{'n':>3}" + "".join(f"{'B=' + str(b):>9}" for b in budgets)
              + f"{'median dE(2000)':>18}")
        for opt in OPTIMIZERS + ['SPSA']:
            s = [r for r in sub if r['opt'] == opt]
            if not s:
                continue
            counts = dict.fromkeys(budgets, 0)
            dE = []
            for r in s:
                bsf = np.minimum.accumulate(np.load(os.path.join(out, 'hist', r['hist_file'])))
                for b in budgets:
                    counts[b] += bsf[min(b, len(bsf)) - 1] - E_ED < CONV_TOL
                dE.append(bsf[min(2000, len(bsf)) - 1] - E_ED)
            print(f"{opt:<12}{len(s):>3}" + "".join(f"{counts[b]:>9}" for b in budgets)
                  + f"{np.median(dE):>18.2e}")
    print()


def fd_gradient(f, x, h=1e-6):
    g = np.zeros(len(x))
    for i in range(len(x)):
        e = np.zeros(len(x))
        e[i] = h
        g[i] = (f(x + e) - f(x - e)) / (2 * h)
    return g


def nm_check(out, engine):
    # Nelder-Mead runs that end in mixed states: rerun them, compute the gradient at
    # the end point and continue from there with L-BFGS-B (Table 4).
    ctx = setup(2)
    sel = [r for r in load_runs(out) if 'E' in r and r['opt'] == 'Nelder-Mead' and
           r['depth'] == 2 and r['protocol'] == 'free' and
           r['dE'] >= CONV_TOL and r['var_H'] > 1e-3]
    print(f"Nelder-Mead runs ending in mixed states: {len(sel)}")
    print(f"{'range':>8}{'seed':>6}{'E_term':>10}{'w_term':>8}{'|grad|':>10}"
          f"{'E_pol':>10}{'w_pol':>8}{'N_pol':>7}")
    for r in sel:
        job = dict(init=r['init'], protocol='replay', depth=2, opt='Nelder-Mead',
                   seed=r['seed'], engine=engine, out=out)
        x = optimize(job, ctx)['best']['x']
        g = fd_gradient(ctx['fast'].energy, x)
        pol = minimize(ctx['fast'].energy, x, method='L-BFGS-B',
                       options={'maxiter': 5000, 'ftol': 1e-15, 'gtol': 1e-10})
        psi = ctx['fast'].state(pol.x)
        w = float(np.sum(psi[ctx['phys']]**2))
        print(f"{r['init']:>8}{r['seed']:>6}{r['E']:>10.5f}{r['w_phys']:>8.3f}"
              f"{np.linalg.norm(g):>10.2e}{pol.fun:>10.5f}{w:>8.3f}{pol.nfev:>7}")


# Appendix B

def exact_reference():
    ctx = setup(2)
    return float(np.linalg.eigvalsh(ctx['H_mat'][np.ix_(ctx['phys'], ctx['phys'])])[0])


def ed_weights():
    # (1p,1n) ground state in the subspace of the ansatz (p down, n down)
    ctx = setup(2)
    labels = ['p0 n0', 'p0 n1', 'p1 n0', 'p1 n1']        # p0: proton on site 0
    idx = [(1 << p) | (1 << n) for p in (1, 5) for n in (3, 7)]
    vals, vecs = np.linalg.eigh(ctx['H_mat'][np.ix_(idx, idx)])
    print(f"ED ground state in the spin-down subspace: E = {vals[0]:.10f} MeV")
    for lab, amp in zip(labels, vecs[:, 0]):
        print(f"  {lab}: weight {amp**2:.6f}")


def mean_std(v):
    v = np.asarray(v, dtype=float)
    if len(v) == 0:
        return '-', None, None
    sd = float(v.std(ddof=1)) if len(v) > 1 else 0.0
    return f"{v.mean():.0f} +- {sd:.0f}", float(v.mean()), sd


def appendix_b(out):
    e_ref = exact_reference()
    print(f"exact (1p,1n) minimum = {e_ref:.10f} MeV, E_ED constant = {E_ED} MeV, "
          f"difference = {e_ref - E_ED:+.3e} MeV")
    ed_weights()

    with open(os.path.join(out, 'runs.jsonl'), encoding='utf-8') as f:
        raw = [json.loads(line) for line in f if line.strip()]
    keys = Counter(job_key(r) for r in raw)
    print(f"\nruns.jsonl: {len(raw)} lines, {len(keys)} unique runs, "
          f"{sum(c - 1 for c in keys.values())} duplicate lines")
    groups = Counter((r['init'], r['protocol'], r['depth'], r['opt']) for r in raw)
    for g, c in sorted(groups.items(), key=str):
        if g[0] not in INITS or c != 20:
            print(f"  not in the reported set or not 20 runs: {g} -> {c}")

    rows = [r for r in load_runs(out) if 'E' in r and r['depth'] == 2]
    cats = ['conv', 's21', 's01', 'sother', 'mixed']
    b1, b2 = [], []

    print("\nTABLE B1  (mean +- sample std; N_10keV over converged runs, N_term over all runs)")
    print(f"{'range':<9}{'optimizer':<12}{'conv':>6}{'N_10keV':>16}{'N_term':>18}"
          f"{'median|dE|':>13}{'max|dE|':>11}{'N10 changed':>13}")
    for init in INITS:
        for opt in OPTIMIZERS + ['SPSA']:
            proto = 'fixed' if opt == 'SPSA' else 'free'
            s = [r for r in rows if r['init'] == init and r['opt'] == opt and r['protocol'] == proto]
            if not s:
                continue
            conv = [r for r in s if classify(r) == 'conv']
            n10_txt, n10_m, n10_s = mean_std([r['evals_to_10keV'] for r in conv])
            if opt == 'SPSA':
                nt_txt, nt_m, nt_s = '-', None, None          # SPSA always stops at the budget
            else:
                nt_txt, nt_m, nt_s = mean_std([r['nfev'] for r in s])
            dev = [r['E'] - e_ref for r in conv]
            changed = 0                                       # N_10keV with the exact reference
            for r in conv:
                h = np.load(os.path.join(out, 'hist', r['hist_file']))
                hit = np.nonzero(np.minimum.accumulate(h) < e_ref + CONV_TOL)[0]
                changed += int(hit[0] + 1) != r['evals_to_10keV']
            med = f"{np.median(dev):.2e}" if dev else '-'
            mx = f"{np.max(dev):.2e}" if dev else '-'
            print(f"{init:<9}{opt:<12}{len(conv):>3}/{len(s):<2}{n10_txt:>16}{nt_txt:>18}"
                  f"{med:>13}{mx:>11}{changed:>13}")
            b1.append(dict(init=init, opt=opt, n=len(s), converged=len(conv),
                           n10_mean=n10_m, n10_std=n10_s, nterm_mean=nt_m, nterm_std=nt_s,
                           dE_median=float(np.median(dev)) if dev else None,
                           dE_max=float(np.max(dev)) if dev else None))
            cnt = Counter(classify(r) for r in s)
            b2.append(dict(init=init, opt=opt, n=len(s), **{c: cnt.get(c, 0) for c in cats}))

    print("\nTABLE B2  (final states; deterministic: free protocol, SPSA: B = 2000)")
    print(f"{'range':<9}{'optimizer':<12}" + "".join(f"{c:>8}" for c in cats))
    for row in b2:
        print(f"{row['init']:<9}{row['opt']:<12}" + "".join(f"{row[c]:>8}" for c in cats))

    det = [row for row in b2 if row['opt'] != 'SPSA']
    print("\ncheck, deterministic optimizers over four ranges:")
    print("  not converged: " + str(sum(r['n'] - r['conv'] for r in det)) + ", "
          + ", ".join(f"{c} {sum(r[c] for r in det)}" for c in cats[1:]))
    nm = [r for r in det if r['opt'] == 'Nelder-Mead']
    print(f"  Nelder-Mead: s21+s01 {sum(r['s21'] + r['s01'] for r in nm)}, "
          f"mixed {sum(r['mixed'] for r in nm)}")

    for name, data in (('appendix_b1.csv', b1), ('appendix_b2.csv', b2)):
        with open(os.path.join(out, name), 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(data[0]))
            w.writeheader()
            w.writerows(data)
    print(f"\nwritten: appendix_b1.csv, appendix_b2.csv in {out}")


def run_all(nseeds, workers, engine):
    out = f"csj_out_{engine}"
    check_setup(out)
    jobs = (make_jobs(nseeds, out, engine)
            + make_jobs(nseeds, out, engine, inits=['narrow', 'pm_pi'], depth=1)
            + make_jobs(nseeds, out, engine, opts=['SPSA'], protocol='fixed',
                        budget=SPSA_BUDGET))
    execute(jobs, out, workers)
    summary(out)
    budget_report(out, engine)
    nm_check(out, engine)
    print(f"\nfiles in {out}: runs.jsonl, runs_flat.csv, summary_stats.csv, hist/")


if __name__ == '__main__':
    ap = argparse.ArgumentParser(description="VQE optimizer benchmark, lattice deuteron")
    ap.add_argument('--seeds', type=int, default=20)
    ap.add_argument('--workers', type=int, default=1)
    ap.add_argument('--engine', choices=['numpy', 'qiskit'], default='numpy')
    ap.add_argument('--appendix', action='store_true',
                    help='only print the Appendix B tables from saved runs')
    in_notebook = 'ipykernel' in sys.modules or 'spyder_kernels' in sys.modules
    # in Jupyter or Spyder: [] runs everything, ['--appendix'] only prints Appendix B
    NOTEBOOK_ARGS = ['--appendix']
    args = ap.parse_args(NOTEBOOK_ARGS if in_notebook else None)
    if args.appendix:
        appendix_b(f"csj_out_{args.engine}")
    else:
        run_all(args.seeds, 1 if in_notebook else args.workers, args.engine)

exact (1p,1n) minimum = -2.2245657565 MeV, E_ED constant = -2.224566 MeV, difference = +2.435e-07 MeV
ED ground state in the spin-down subspace: E = -2.2245657565 MeV
  p0 n0: weight 0.270987
  p0 n1: weight 0.229013
  p1 n0: weight 0.229013
  p1 n1: weight 0.270987

runs.jsonl: 680 lines, 680 unique runs, 0 duplicate lines

TABLE B1  (mean +- sample std; N_10keV over converged runs, N_term over all runs)
range    optimizer     conv         N_10keV            N_term   median|dE|    max|dE|  N10 changed
narrow   COBYLA       20/20       106 +- 22      6139 +- 6890     5.20e-11   1.08e-07            0
narrow   Nelder-Mead  20/20    4043 +- 2724      7091 +- 2782     1.82e-14   3.01e-03            0
narrow   Powell       20/20      624 +- 217       1639 +- 340     1.91e-07   5.18e-06            0
narrow   SLSQP        20/20       150 +- 19         417 +- 45     2.21e-10   4.36e-10            0
narrow   L-BFGS-B     20/20       193 +- 40         676 +- 98     4.70e-13   3.60e-12           